<a href="https://colab.research.google.com/github/Aws-Abdullah/NLP-Project/blob/main/Phase1_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Arabic Abstractive Text Summarization - Phase 1
## Data Loading and Preprocessing

This notebook loads four Arabic summarization datasets, unifies them, cleans the text, and prepares tokenized sequences ready for the Seq2Seq model in Phase 2.

## Setup

In [ ]:
!pip install pyarrow -q

In [ ]:
import pandas as pd
import numpy as np
import json
import re
import pickle

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

import matplotlib.pyplot as plt

## Load the four datasets

Each dataset is loaded into a DataFrame with two columns: `text` (input article) and `summarizer` (target summary). Column names and file formats differ across sources, so each needs its own loader.

### AraSum (TSV format, no header)

In [ ]:
df_arasum = pd.read_csv('arasum.csv', sep='\t', header=None, names=['id', 'text', 'summarizer'])
df_arasum = df_arasum[['text', 'summarizer']]
print('AraSum:', df_arasum.shape)
df_arasum.head()

### SumArabic test set (CSV)

In [ ]:
df_sumtest = pd.read_csv('sumtest.csv')
df_sumtest = df_sumtest[['text', 'summarizer']]
print('SumTest:', df_sumtest.shape)
df_sumtest.head()

### SumArabic JSONL files (train, valid, test)

These use `headline` as the summary field, so we rename it to `summarizer`.

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            rows.append({
                'text': item.get('text', ''),
                'summarizer': item.get('headline', '')
            })
    return pd.DataFrame(rows)

df_sumar_train = load_jsonl('sumartrain.jsonl')
df_sumar_valid = load_jsonl('sumarvalid.jsonl')
df_sumar_test  = load_jsonl('sumartest.jsonl')

df_sumarabic = pd.concat([df_sumar_train, df_sumar_valid, df_sumar_test], ignore_index=True)
print('SumArabic:', df_sumarabic.shape)
df_sumarabic.head()

### Egyptian Arabic (parquet)

Column names here are unpredictable, so we auto-detect the text and summary columns by checking for common names. If detection fails, we fall back to the first two columns.

In [ ]:
df_egy = pd.read_parquet('Egy.parquet')
print('Original columns:', df_egy.columns.tolist())

cols = df_egy.columns.tolist()
text_col = next((c for c in cols if c.lower() in ['text', 'article', 'content']), cols[0])
summary_col = next((c for c in cols if c.lower() in ['summary', 'headline', 'title']), cols[1])

print(f'Using: {text_col} -> text, {summary_col} -> summarizer')

df_egy = df_egy.rename(columns={text_col: 'text', summary_col: 'summarizer'})
df_egy = df_egy[['text', 'summarizer']]
print('Egyptian:', df_egy.shape)
df_egy.head()

## Merge all datasets

In [ ]:
df = pd.concat([df_arasum, df_sumtest, df_sumarabic, df_egy], ignore_index=True)
print('Total rows before cleaning:', df.shape[0])

## Text cleaning

Arabic text needs several cleaning steps beyond just removing English characters:

- **Diacritics (tashkeel)**: marks like ً ٌ ٍ َ ُ ِ ّ ْ that indicate pronunciation. These add noise and are optional in modern Arabic writing.
- **Tatweel (ـ)**: the elongation character used for visual alignment, not meaning.
- **Non-Arabic characters**: English letters, numbers, punctuation, symbols.
- **Extra whitespace**: normalized to single spaces.

All cleaning is applied to both articles and summaries.

In [ ]:
# Drop nulls and convert to string
df = df.dropna(subset=['text', 'summarizer'])
df['text'] = df['text'].astype(str)
df['summarizer'] = df['summarizer'].astype(str)

# Compile regex patterns once
diacritics = re.compile(r'[\u064B-\u065F\u0670]')   # tashkeel range + alef superscript
tatweel = re.compile(r'\u0640')                        # elongation character
non_arabic = re.compile(r'[^\u0600-\u06FF\s]')       # keep only Arabic block + whitespace
extra_spaces = re.compile(r'\s+')

def clean_arabic(text):
    text = diacritics.sub('', text)
    text = tatweel.sub('', text)
    text = non_arabic.sub(' ', text)
    text = extra_spaces.sub(' ', text)
    return text.strip()

df['text'] = df['text'].apply(clean_arabic)
df['summarizer'] = df['summarizer'].apply(clean_arabic)

print('Cleaning done')

## Arabic letter normalization

Different forms of the same letter are unified to reduce vocabulary size and handle writing inconsistencies across sources:

- Alef variants (إ أ آ ا) → ا
- Ya variants (ى ي) → ي
- Hamza-carrying letters (ؤ ئ) → ء
- Ta marbuta (ة) → ه

In [ ]:
def normalize_arabic(text):
    text = re.sub('[إأآا]', 'ا', text)
    text = re.sub('ى', 'ي', text)
    text = re.sub('ؤ', 'ء', text)
    text = re.sub('ئ', 'ء', text)
    text = re.sub('ة', 'ه', text)
    return text

df['text'] = df['text'].apply(normalize_arabic)
df['summarizer'] = df['summarizer'].apply(normalize_arabic)

print('Normalization done')

## Filter invalid samples

After cleaning, remove:
- Empty rows (cleaning may have emptied some)
- Articles shorter than 10 words (too short to summarize)
- Summaries shorter than 3 words (not meaningful)
- Duplicate (text, summary) pairs

In [ ]:
df = df[(df['text'] != '') & (df['summarizer'] != '')]

df['text_len'] = df['text'].str.split().str.len()
df['summary_len'] = df['summarizer'].str.split().str.len()

df = df[(df['text_len'] >= 10) & (df['summary_len'] >= 3)]

df = df.drop_duplicates(subset=['text', 'summarizer']).reset_index(drop=True)

print(f'Rows after filtering: {df.shape[0]}')

## Length distributions

Plot article and summary length distributions to choose reasonable `MAX_ARTICLE_LEN` and `MAX_SUMMARY_LEN` values. Too short wastes information; too long wastes memory and padding.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['text_len'].clip(upper=500), bins=50, color='steelblue')
axes[0].axvline(200, color='red', linestyle='--', label='MAX_ARTICLE_LEN=200')
axes[0].set_title('Article lengths (capped at 500)')
axes[0].set_xlabel('Words')
axes[0].legend()

axes[1].hist(df['summary_len'].clip(upper=100), bins=50, color='darkorange')
axes[1].axvline(50, color='red', linestyle='--', label='MAX_SUMMARY_LEN=50')
axes[1].set_title('Summary lengths (capped at 100)')
axes[1].set_xlabel('Words')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Article: median={df["text_len"].median():.0f}, 95th pct={df["text_len"].quantile(0.95):.0f}')
print(f'Summary: median={df["summary_len"].median():.0f}, 95th pct={df["summary_len"].quantile(0.95):.0f}')

## Add sos and eos tokens to summaries

These special tokens mark the start and end of each summary. The decoder uses `sos` to start generation and produces `eos` when it decides the summary is complete.

In [ ]:
df['summarizer'] = 'sos ' + df['summarizer'] + ' eos'
df.head(3)

## Train / validation / test split (80 / 10 / 10)

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, shuffle=True)

print(f'Train: {train_df.shape[0]}')
print(f'Val:   {val_df.shape[0]}')
print(f'Test:  {test_df.shape[0]}')

## Tokenization

Fit the tokenizer on **both articles and summaries** so that summary-specific words (including `sos` and `eos`) are in the vocabulary. We cap vocabulary size at 30k most frequent words; everything else becomes `<OOV>`.

In [ ]:
VOCAB_SIZE = 30000

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(train_df['text'])
tokenizer.fit_on_texts(train_df['summarizer'])

print(f'Vocab size: {len(tokenizer.word_index)}')
print(f'sos_id = {tokenizer.word_index["sos"]}')
print(f'eos_id = {tokenizer.word_index["eos"]}')

## Convert to sequences and pad

In [ ]:
MAX_ARTICLE_LEN = 400
MAX_SUMMARY_LEN = 50

X_train_raw = tokenizer.texts_to_sequences(train_df['text'])
X_val_raw   = tokenizer.texts_to_sequences(val_df['text'])
X_test_raw  = tokenizer.texts_to_sequences(test_df['text'])

y_train_raw = tokenizer.texts_to_sequences(train_df['summarizer'])
y_val_raw   = tokenizer.texts_to_sequences(val_df['summarizer'])
y_test_raw  = tokenizer.texts_to_sequences(test_df['summarizer'])

X_train = pad_sequences(X_train_raw, maxlen=MAX_ARTICLE_LEN, padding='post', truncating='post')
X_val   = pad_sequences(X_val_raw,   maxlen=MAX_ARTICLE_LEN, padding='post', truncating='post')
X_test  = pad_sequences(X_test_raw,  maxlen=MAX_ARTICLE_LEN, padding='post', truncating='post')

y_train = pad_sequences(y_train_raw, maxlen=MAX_SUMMARY_LEN, padding='post', truncating='post')
y_val   = pad_sequences(y_val_raw,   maxlen=MAX_SUMMARY_LEN, padding='post', truncating='post')
y_test  = pad_sequences(y_test_raw,  maxlen=MAX_SUMMARY_LEN, padding='post', truncating='post')

print(f'X_train: {X_train.shape}')
print(f'y_train: {y_train.shape}')
print(f'X_val:   {X_val.shape}')
print(f'y_val:   {y_val.shape}')
print(f'X_test:  {X_test.shape}')
print(f'y_test:  {y_test.shape}')

## (Optional) Save outputs for Phase 2

If you want to run Phase 2 in a separate session, save the arrays and tokenizer here. By default these save to the Colab session (lost when the runtime disconnects). To persist them, mount Google Drive and change the paths to a Drive folder.

In [ ]:
np.save('X_train.npy', X_train)
np.save('X_val.npy', X_val)
np.save('X_test.npy', X_test)
np.save('y_train.npy', y_train)
np.save('y_val.npy', y_val)
np.save('y_test.npy', y_test)

with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

train_df.to_pickle('train_df.pkl')
val_df.to_pickle('val_df.pkl')
test_df.to_pickle('test_df.pkl')

print('Saved to Colab session.')
print('To persist across sessions, mount Drive and save to a Drive folder instead.')

---

**Ready for Phase 2.** The following variables are in memory and can be used directly by Phase 2 cells pasted below, or reloaded from the saved files:

- `X_train`, `X_val`, `X_test`, `y_train`, `y_val`, `y_test`
- `tokenizer`
- `train_df`, `val_df`, `test_df`